In [1]:
import os
os.environ["HF_ALLOW_CODE_EVAL"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [2]:
from datasets import load_dataset
from evaluate import load
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from tqdm import tqdm

# Load HumanEval dataset
human_eval = load_dataset("openai_humaneval")['test']

# Load code evaluation metric
code_eval_metric = load("code_eval")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

openai_humaneval/test-00000-of-00001.par(…):   0%|          | 0.00/83.9k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

In [ ]:
"""
Bayesian Prompt Ensembles for Uncertainty-Aware Self-Correcting Code Generation

Research Questions:
1. How well do uncertainty signals correlate with code correctness?
2. What uncertainty thresholds trigger effective self-corrections?
3. Can lightweight self-correction reduce human edits while staying responsive?
"""

import os
os.environ["HF_ALLOW_CODE_EVAL"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from evaluate import load
import numpy as np
from scipy.optimize import minimize
from scipy.special import softmax
from scipy.stats import spearmanr, pearsonr, entropy
from tqdm import tqdm
import json
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Tuple
import warnings
warnings.filterwarnings('ignore')

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


class BayesianPromptEnsemble:
    """Implements BayesPE for code generation with uncertainty estimation"""

    def __init__(self, prompt_templates: List[str]):
        self.prompt_templates = prompt_templates
        self.n_prompts = len(prompt_templates)
        # Initialize uniform weights
        self.weights = np.ones(self.n_prompts) / self.n_prompts

    def optimize_weights(self, validation_results: List[Dict]):
        """
        Optimize weights using BayesPE objective (Eq. 4 from paper)
        Maximizes: sum(w_i * log p(y*|a_i, x)) - sum(w_i * log w_i)
        """
        def objective(w):
            w = softmax(w)
            likelihood = 0
            for result in validation_results:
                for i, prob in enumerate(result['prompt_probs']):
                    likelihood += w[i] * np.log(prob + 1e-10)
            entropy_term = -np.sum(w * np.log(w + 1e-10))
            return -(likelihood + entropy_term)

        w0 = np.zeros(self.n_prompts)
        result = minimize(objective, w0, method='L-BFGS-B')
        self.weights = softmax(result.x)
        print(f"Optimized BayesPE weights: {self.weights}")

    def compute_uncertainty(self, prompt_outputs: List[Dict]) -> Dict:
        """
        Compute multiple uncertainty metrics from prompt ensemble

        Returns:
            - predictive_entropy: Total uncertainty
            - mutual_information: Epistemic (model) uncertainty
            - expected_entropy: Aleatoric (data) uncertainty
            - prompt_variance: Variance in token probabilities across prompts
            - max_disagreement: Maximum difference in probabilities
        """
        # Extract average probabilities from each prompt
        prompt_probs = np.array([out['avg_token_prob'] for out in prompt_outputs])

        # Weighted average probability
        weighted_avg_prob = np.sum(self.weights * prompt_probs)

        # Predictive entropy (total uncertainty)
        predictive_entropy = -weighted_avg_prob * np.log(weighted_avg_prob + 1e-10)

        # Expected entropy (aleatoric uncertainty)
        expected_entropy = -np.sum(
            self.weights * prompt_probs * np.log(prompt_probs + 1e-10)
        )

        # Mutual information (epistemic uncertainty)
        mutual_information = predictive_entropy - expected_entropy

        # Prompt variance (disagreement measure)
        prompt_variance = np.var(prompt_probs)

        # Maximum disagreement between prompts
        max_disagreement = np.max(prompt_probs) - np.min(prompt_probs)

        # Token-level entropy (average across each sequence separately)
        # Since sequences have different lengths, compute per-sequence then average
        sequence_entropies = []
        for out in prompt_outputs:
            token_probs = np.array(out['token_probs'])
            # Entropy of the token probability distribution for this sequence
            if len(token_probs) > 0:
                seq_entropy = entropy(token_probs + 1e-10)
                sequence_entropies.append(seq_entropy)

        avg_token_entropy = np.mean(sequence_entropies) if sequence_entropies else 0.0

        return {
            'predictive_entropy': float(predictive_entropy),
            'mutual_information': float(mutual_information),
            'expected_entropy': float(expected_entropy),
            'prompt_variance': float(prompt_variance),
            'max_disagreement': float(max_disagreement),
            'avg_token_entropy': float(avg_token_entropy),
            'weighted_avg_prob': float(weighted_avg_prob)
        }


class CodeGenerator:
    """Handles code generation with uncertainty estimation"""

    def __init__(self, model_name: str):
        print(f"Loading model: {model_name}")
        self.model_name = model_name
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_name,
            trust_remote_code=True
        )
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype="auto",  # Auto-detect best dtype
            device_map="auto",
            trust_remote_code=True
        )
        self.model.eval()

        # Set special tokens
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        if self.tokenizer.pad_token_id is None:
            self.tokenizer.pad_token_id = self.tokenizer.eos_token_id

    def generate_with_uncertainty(
        self,
        prompt: str,
        max_new_tokens: int = 256,
        temperature: float = 0.2
    ) -> Dict:
        """
        Generate code and compute token-level probabilities for uncertainty
        """
        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=2048
        ).to(device)

        with torch.no_grad():
            outputs = self.model.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_new_tokens=max_new_tokens,
                return_dict_in_generate=True,
                output_scores=True,
                do_sample=True,
                temperature=temperature,
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id
            )

        # Extract generated tokens (excluding prompt)
        generated_tokens = outputs.sequences[0][inputs["input_ids"].shape[1]:]
        generated_text = self.tokenizer.decode(generated_tokens, skip_special_tokens=True)

        # Compute token probabilities
        token_probs = []
        for i, logits in enumerate(outputs.scores):
            if i >= len(generated_tokens):
                break
            probs = F.softmax(logits[0], dim=-1)
            token_id = generated_tokens[i].item()
            token_prob = probs[token_id].item()
            token_probs.append(token_prob)

        # Average probability (geometric mean in log space)
        if len(token_probs) > 0:
            avg_token_prob = np.exp(np.mean(np.log(np.array(token_probs) + 1e-10)))
        else:
            avg_token_prob = 0.01  # Default low probability for empty generation

        return {
            'text': generated_text,
            'token_probs': token_probs,
            'avg_token_prob': avg_token_prob
        }

    def self_correct(
        self,
        original_prompt: str,
        generated_code: str,
        error_msg: str = None
    ) -> str:
        """
        Self-correction via repair prompting
        """
        if error_msg:
            repair_prompt = f"""{original_prompt}

Previous attempt:
```python
{generated_code}
```

Error: {error_msg}

Please fix the code above to resolve this error:
```python"""
        else:
            repair_prompt = f"""{original_prompt}

Previous attempt:
```python
{generated_code}
```

Please review and improve the code above:
```python"""

        output = self.generate_with_uncertainty(
            repair_prompt,
            max_new_tokens=256,
            temperature=0.1  # Lower temperature for corrections
        )
        return output['text']


def create_prompt_templates(base_problem: str) -> List[str]:
    """Create semantically equivalent prompt variations"""
    templates = [
        f"{base_problem}",
        f"Complete the following Python function:\n{base_problem}",
        f"Implement this function according to its docstring:\n{base_problem}",
        f"Write the code for this function:\n{base_problem}",
        f"Your task: complete this Python function.\n{base_problem}",
    ]
    return templates


def evaluate_code(code: str, test: str, timeout: float = 3.0) -> Dict:
    """
    Execute code with test cases safely
    Returns: {'passed': bool, 'error': str or None}
    """
    import subprocess
    import tempfile

    full_code = code + "\n" + test

    with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
        f.write(full_code)
        temp_file = f.name

    try:
        result = subprocess.run(
            ['python', temp_file],
            capture_output=True,
            text=True,
            timeout=timeout
        )
        os.unlink(temp_file)

        if result.returncode == 0:
            return {'passed': True, 'error': None}
        else:
            return {'passed': False, 'error': result.stderr}
    except subprocess.TimeoutExpired:
        os.unlink(temp_file)
        return {'passed': False, 'error': 'Timeout'}
    except Exception as e:
        if os.path.exists(temp_file):
            os.unlink(temp_file)
        return {'passed': False, 'error': str(e)}


def run_experiment(
    model_name: str,
    n_problems: int = 50,
    n_validation: int = 5,
    uncertainty_threshold: float = 0.5,
    enable_self_correction: bool = True
) -> Dict:
    """
    Main experiment: Test BayesPE with self-correction on HumanEval

    Returns comprehensive results for research questions
    """
    print(f"\n{'='*80}")
    print(f"Experiment: {model_name}")
    print(f"{'='*80}\n")

    # Load dataset
    print("Loading HumanEval dataset...")
    human_eval = load_dataset("openai_humaneval")['test']
    human_eval = list(human_eval)

    # Initialize generator
    generator = CodeGenerator(model_name)

    # Phase 1: Optimize BayesPE on validation set
    print("\nPhase 1: BayesPE Weight Optimization")
    print("-" * 80)

    validation_set = human_eval[:n_validation]
    validation_results = []

    for problem in tqdm(validation_set, desc="Validation"):
        prompts = create_prompt_templates(problem['prompt'])

        # Generate with each prompt
        prompt_probs = []
        for prompt in prompts:
            output = generator.generate_with_uncertainty(prompt)
            result = evaluate_code(output['text'], problem['test'])
            # Use correctness as probability signal
            prob = 1.0 if result['passed'] else 0.1
            prompt_probs.append(prob)

        validation_results.append({'prompt_probs': prompt_probs})

    # Initialize and optimize BayesPE
    prompts_template = create_prompt_templates("")
    bayespe = BayesianPromptEnsemble(prompts_template)
    bayespe.optimize_weights(validation_results)

    # Phase 2: Test with uncertainty estimation and self-correction
    print("\nPhase 2: Testing with Uncertainty Estimation")
    print("-" * 80)

    test_set = human_eval[n_validation:n_validation + n_problems]
    results = []

    # Progress tracking with time estimates
    import time
    start_time = time.time()

    for prob_idx, problem in enumerate(tqdm(test_set, desc="Testing", unit="problem")):
        prompts = create_prompt_templates(problem['prompt'])

        # Generate with each prompt
        prompt_outputs = []
        for prompt in prompts:
            output = generator.generate_with_uncertainty(prompt)
            prompt_outputs.append(output)

        # Compute uncertainty
        try:
            uncertainty_metrics = bayespe.compute_uncertainty(prompt_outputs)
        except Exception as e:
            print(f"\n⚠️  Warning: Could not compute uncertainty for problem {prob_idx}: {e}")
            # Use default uncertainty values
            uncertainty_metrics = {
                'predictive_entropy': 0.5,
                'mutual_information': 0.5,
                'expected_entropy': 0.5,
                'prompt_variance': 0.1,
                'max_disagreement': 0.2,
                'avg_token_entropy': 0.5,
                'weighted_avg_prob': 0.5
            }

        # Select best candidate (weighted by BayesPE)
        weighted_scores = [
            w * out['avg_token_prob']
            for w, out in zip(bayespe.weights, prompt_outputs)
        ]
        best_idx = np.argmax(weighted_scores)
        best_output = prompt_outputs[best_idx]

        # Initial evaluation
        initial_result = evaluate_code(best_output['text'], problem['test'])

        # Self-correction policy
        corrected = False
        correction_improved = False

        if (enable_self_correction and
            not initial_result['passed'] and
            uncertainty_metrics['mutual_information'] > uncertainty_threshold):

            # Trigger self-correction
            try:
                corrected_code = generator.self_correct(
                    problem['prompt'],
                    best_output['text'],
                    initial_result['error']
                )

                corrected_result = evaluate_code(corrected_code, problem['test'])
                corrected = True
                correction_improved = corrected_result['passed'] and not initial_result['passed']

                final_result = corrected_result
                final_code = corrected_code
            except Exception as e:
                print(f"\n⚠️  Warning: Self-correction failed for problem {prob_idx}: {e}")
                final_result = initial_result
                final_code = best_output['text']
        else:
            final_result = initial_result
            final_code = best_output['text']

        # Store results
        results.append({
            'task_id': problem['task_id'],
            'initial_passed': initial_result['passed'],
            'final_passed': final_result['passed'],
            'corrected': corrected,
            'correction_improved': correction_improved,
            'uncertainty': uncertainty_metrics,
            'initial_code': best_output['text'],
            'final_code': final_code,
            'all_prompt_passed': [
                evaluate_code(out['text'], problem['test'])['passed']
                for out in prompt_outputs
            ]
        })

        # Print time estimate every 5 problems
        if (prob_idx + 1) % 5 == 0:
            elapsed = time.time() - start_time
            avg_time = elapsed / (prob_idx + 1)
            remaining = (len(test_set) - (prob_idx + 1)) * avg_time
            print(f"  ⏱️  Avg: {avg_time:.1f}s/problem | Remaining: {remaining/60:.1f} min")

    # Compute pass@k metrics
    print("\nPhase 3: Computing Pass@k Metrics")
    print("-" * 80)

    # Pass@1 (best prompt)
    pass_at_1_initial = np.mean([r['initial_passed'] for r in results])
    pass_at_1_final = np.mean([r['final_passed'] for r in results])

    # Pass@5 (any prompt succeeds)
    pass_at_5 = np.mean([any(r['all_prompt_passed']) for r in results])

    # Correction statistics
    n_corrections = sum(r['corrected'] for r in results)
    n_successful_corrections = sum(r['correction_improved'] for r in results)
    correction_success_rate = (
        n_successful_corrections / n_corrections if n_corrections > 0 else 0.0
    )

    print(f"\nResults:")
    print(f"  Pass@1 (initial):     {pass_at_1_initial:.3f}")
    print(f"  Pass@1 (final):       {pass_at_1_final:.3f}")
    print(f"  Pass@5 (ensemble):    {pass_at_5:.3f}")
    print(f"  Corrections made:     {n_corrections}/{len(results)}")
    print(f"  Correction success:   {correction_success_rate:.3f}")

    return {
        'model_name': model_name,
        'results': results,
        'pass_at_1_initial': float(pass_at_1_initial),
        'pass_at_1_final': float(pass_at_1_final),
        'pass_at_5': float(pass_at_5),
        'correction_success_rate': float(correction_success_rate),
        'n_corrections': int(n_corrections),
        'bayespe_weights': bayespe.weights.tolist()
    }


def analyze_uncertainty_correlation(results: List[Dict]) -> Dict:
    """
    Research Question 1: Analyze correlation between uncertainty and correctness
    """
    print("\n" + "="*80)
    print("RESEARCH QUESTION 1: Uncertainty-Correctness Correlation")
    print("="*80)

    # Extract data
    correctness = np.array([r['initial_passed'] for r in results])

    uncertainty_metrics = {
        'mutual_information': np.array([r['uncertainty']['mutual_information'] for r in results]),
        'predictive_entropy': np.array([r['uncertainty']['predictive_entropy'] for r in results]),
        'prompt_variance': np.array([r['uncertainty']['prompt_variance'] for r in results]),
        'max_disagreement': np.array([r['uncertainty']['max_disagreement'] for r in results]),
        'avg_token_entropy': np.array([r['uncertainty']['avg_token_entropy'] for r in results])
    }

    correlations = {}

    for metric_name, metric_values in uncertainty_metrics.items():
        # Spearman correlation (for monotonic relationships)
        spearman_corr, spearman_p = spearmanr(metric_values, correctness)

        # Pearson correlation (for linear relationships)
        pearson_corr, pearson_p = pearsonr(metric_values, correctness)

        correlations[metric_name] = {
            'spearman': float(spearman_corr),
            'spearman_p': float(spearman_p),
            'pearson': float(pearson_corr),
            'pearson_p': float(pearson_p)
        }

        print(f"\n{metric_name}:")
        print(f"  Spearman: {spearman_corr:+.3f} (p={spearman_p:.4f})")
        print(f"  Pearson:  {pearson_corr:+.3f} (p={pearson_p:.4f})")

    # Find best uncertainty metric
    best_metric = max(
        correlations.items(),
        key=lambda x: abs(x[1]['spearman'])
    )

    print(f"\n🏆 Best uncertainty metric: {best_metric[0]}")
    print(f"   Correlation: {best_metric[1]['spearman']:+.3f}")

    return {
        'correlations': correlations,
        'best_metric': best_metric[0],
        'uncertainty_metrics': uncertainty_metrics,
        'correctness': correctness
    }


def analyze_optimal_threshold(results: List[Dict]) -> Dict:
    """
    Research Question 2: Find optimal uncertainty threshold for corrections
    """
    print("\n" + "="*80)
    print("RESEARCH QUESTION 2: Optimal Uncertainty Threshold")
    print("="*80)

    # Extract data for problems that were corrected
    corrected_results = [r for r in results if r['corrected']]

    if len(corrected_results) == 0:
        print("No corrections were made - cannot analyze thresholds")
        return {}

    mi_values = np.array([r['uncertainty']['mutual_information'] for r in corrected_results])
    improvements = np.array([r['correction_improved'] for r in corrected_results])

    # Try different thresholds
    thresholds = np.percentile(
        [r['uncertainty']['mutual_information'] for r in results],
        [10, 25, 50, 75, 90]
    )

    threshold_analysis = []

    for threshold in thresholds:
        # Simulate applying this threshold
        would_correct = np.array([
            r['uncertainty']['mutual_information'] > threshold
            for r in results
        ])

        n_corrections = np.sum(would_correct)

        if n_corrections > 0:
            # Among corrected, how many improved?
            actually_corrected = [r for r in results if r['corrected']]
            relevant_corrections = [
                r['correction_improved']
                for r in results
                if r['uncertainty']['mutual_information'] > threshold and r['corrected']
            ]

            precision = np.mean(relevant_corrections) if relevant_corrections else 0.0
        else:
            precision = 0.0

        threshold_analysis.append({
            'threshold': float(threshold),
            'n_corrections': int(n_corrections),
            'precision': float(precision)
        })

        print(f"\nThreshold: {threshold:.4f}")
        print(f"  Would trigger: {n_corrections} corrections")
        print(f"  Precision:     {precision:.3f}")

    # Find optimal (balance corrections and precision)
    optimal = max(threshold_analysis, key=lambda x: x['precision'])

    print(f"\n🎯 Optimal threshold: {optimal['threshold']:.4f}")
    print(f"   Precision: {optimal['precision']:.3f}")
    print(f"   Corrections: {optimal['n_corrections']}")

    return {
        'threshold_analysis': threshold_analysis,
        'optimal_threshold': optimal
    }


def analyze_self_correction_policy(results: List[Dict]) -> Dict:
    """
    Research Question 3: Evaluate self-correction effectiveness
    """
    print("\n" + "="*80)
    print("RESEARCH QUESTION 3: Self-Correction Policy Effectiveness")
    print("="*80)

    total_problems = len(results)
    n_initially_wrong = sum(not r['initial_passed'] for r in results)
    n_corrections = sum(r['corrected'] for r in results)
    n_improvements = sum(r['correction_improved'] for r in results)
    n_final_correct = sum(r['final_passed'] for r in results)

    # Metrics
    improvement_rate = n_improvements / n_initially_wrong if n_initially_wrong > 0 else 0
    correction_precision = n_improvements / n_corrections if n_corrections > 0 else 0
    overall_improvement = (n_final_correct - sum(r['initial_passed'] for r in results)) / total_problems

    print(f"\nCorrection Statistics:")
    print(f"  Initially wrong:       {n_initially_wrong}/{total_problems}")
    print(f"  Corrections triggered: {n_corrections}")
    print(f"  Corrections improved:  {n_improvements}")
    print(f"  Final correct:         {n_final_correct}/{total_problems}")
    print(f"\nEffectiveness:")
    print(f"  Improvement rate:      {improvement_rate:.3f}")
    print(f"  Correction precision:  {correction_precision:.3f}")
    print(f"  Overall improvement:   {overall_improvement:+.3f}")

    # Estimate human edit reduction
    # Assume: correct code = 0 edits, incorrect = 1 edit
    # Self-correction that works = 0 edits needed
    edits_without_correction = n_initially_wrong
    edits_with_correction = n_initially_wrong - n_improvements
    edit_reduction = (edits_without_correction - edits_with_correction) / edits_without_correction if edits_without_correction > 0 else 0

    print(f"\nHuman Edit Reduction:")
    print(f"  Without correction:    {edits_without_correction} edits needed")
    print(f"  With correction:       {edits_with_correction} edits needed")
    print(f"  Reduction:             {edit_reduction:.1%}")

    return {
        'n_corrections': n_corrections,
        'n_improvements': n_improvements,
        'improvement_rate': float(improvement_rate),
        'correction_precision': float(correction_precision),
        'overall_improvement': float(overall_improvement),
        'edit_reduction': float(edit_reduction)
    }


def compare_models(
    model_names: List[str],
    n_problems: int = 50,
    uncertainty_threshold: float = 0.5
):
    """Run experiments on multiple models and compare"""

    all_results = {}
    all_analyses = {}

    for model_name in model_names:
        try:
            # Run experiment
            exp_results = run_experiment(
                model_name=model_name,
                n_problems=n_problems,
                uncertainty_threshold=uncertainty_threshold,
                enable_self_correction=True
            )

            # Analyze
            corr_analysis = analyze_uncertainty_correlation(exp_results['results'])
            thresh_analysis = analyze_optimal_threshold(exp_results['results'])
            correction_analysis = analyze_self_correction_policy(exp_results['results'])

            all_results[model_name] = exp_results
            all_analyses[model_name] = {
                'correlation': corr_analysis,
                'threshold': thresh_analysis,
                'correction': correction_analysis
            }

            # Save individual results
            output_file = f"results_{model_name.replace('/', '_')}.json"
            with open(output_file, 'w') as f:
                json.dump({
                    'experiment': exp_results,
                    'analysis': all_analyses[model_name]
                }, f, indent=2, default=str)

            print(f"\n💾 Saved: {output_file}")

            # Clear GPU memory
            import gc
            torch.cuda.empty_cache()
            gc.collect()

        except Exception as e:
            print(f"\n❌ Error with {model_name}: {e}")
            import traceback
            traceback.print_exc()

    # Print comparison
    print_comparison_summary(all_results, all_analyses)

    return all_results, all_analyses


def print_comparison_summary(all_results: Dict, all_analyses: Dict):
    """Print final comparison table"""
    print("\n" + "="*100)
    print("FINAL COMPARISON: BayesPE Uncertainty-Aware Self-Correction")
    print("="*100)

    print(f"\n{'Model':<40} {'Pass@1':<10} {'Pass@5':<10} {'+Corr':<10} {'Best Metric':<20} {'Corr':<8}")
    print("-" * 100)

    for model_name, results in all_results.items():
        model_short = model_name.split('/')[-1][:38]
        analysis = all_analyses[model_name]

        best_metric = analysis['correlation']['best_metric']
        best_corr = analysis['correlation']['correlations'][best_metric]['spearman']

        print(f"{model_short:<40} "
              f"{results['pass_at_1_initial']:<10.3f} "
              f"{results['pass_at_5']:<10.3f} "
              f"{results['pass_at_1_final']:<10.3f} "
              f"{best_metric[:18]:<20} "
              f"{best_corr:<+8.3f}")

    print("\n" + "="*100)
    print("Legend:")
    print("  Pass@1: Initial pass rate (best prompt)")
    print("  Pass@5: Pass rate with 5 prompts (ensemble)")
    print("  +Corr:  Pass rate after self-correction")
    print("  Corr:   Spearman correlation (uncertainty vs correctness)")


# Main execution
if __name__ == "__main__":
    print("="*80)
    print("BayesPE Research: Uncertainty-Aware Self-Correcting Code Generation")
    print("="*80)

    # Models to test (start with smaller open-source models)
    models = [
        "Qwen/Qwen2.5-Coder-1.5B",  # Start small for testing
        # "Qwen/Qwen2.5-Coder-3B",
        # "deepseek-ai/deepseek-coder-1.3b-base",
    ]

    print(f"\nModels: {models}")
    print(f"Problems: 50 per model")
    print(f"Self-correction: Enabled")
    print("="*80)

    # Run experiments
    all_results, all_analyses = compare_models(
        model_names=models,
        n_problems=5,
        uncertainty_threshold=0.5
    )

    print("\n✅ All experiments complete!")
    print("📊 Results saved to results_*.json files")

Using device: cpu
BayesPE Research: Uncertainty-Aware Self-Correcting Code Generation

Models: ['Qwen/Qwen2.5-Coder-1.5B']
Problems: 50 per model
Self-correction: Enabled

Experiment: Qwen/Qwen2.5-Coder-1.5B

Loading HumanEval dataset...


README.md: 0.00B [00:00, ?B/s]

openai_humaneval/test-00000-of-00001.par(…):   0%|          | 0.00/83.9k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

Loading model: Qwen/Qwen2.5-Coder-1.5B


...

generation_config.json:   0%|          | 0.00/117 [00:00<?, ?B/s]


Phase 1: BayesPE Weight Optimization
--------------------------------------------------------------------------------


Validation: 100%|██████████| 5/5 [47:28<00:00, 569.79s/it]


Optimized BayesPE weights: [0.2 0.2 0.2 0.2 0.2]

Phase 2: Testing with Uncertainty Estimation
--------------------------------------------------------------------------------


Testing:  60%|██████    | 3/5 [23:20<15:59, 479.54s/problem]